Exemplo de teste para 3 períodos e 3 barras

1. Estrutura básica do notebook e dados de brinquedo
Primeiro, um esqueleto mínimo com PuLP e uma microrrede bem pequena para validar:

In [1]:
import pandas as pd
import pulp
import numpy as np

# ============================================================
# 1) LEITURA DO CSV ÚNICO
# ============================================================
arquivo_csv = "dados_microrrede.csv"
df = pd.read_csv(arquivo_csv)

# Padronização básica
df["tipo"] = df["tipo"].astype(str).str.upper().str.strip()

# Horizonte fixo da proposta
T = range(1, 25)   # 24 períodos
Delta_t = 1.0

# ============================================================
# 2) CONJUNTO DE BARRAS N
# ============================================================
# Opção principal: ler linhas tipo BARRA
df_barras = df[df["tipo"] == "BARRA"].copy()

if not df_barras.empty:
    N = sorted(df_barras["i"].dropna().astype(int).unique())
else:
    # fallback: inferir barras de qualquer coluna i/j existente
    barras_i = df["i"].dropna().astype(int).unique() if "i" in df.columns else []
    barras_j = df["j"].dropna().astype(int).unique() if "j" in df.columns else []
    N = sorted(set(barras_i).union(set(barras_j)))

# ============================================================
# 3) CONJUNTOS GS, GW, B
# ============================================================
df_ativos = df[df["tipo"] == "ATIVO"].copy()
df_ativos["ativo"] = df_ativos["ativo"].astype(str).str.lower().str.strip()

GS = set(df_ativos.loc[df_ativos["ativo"] == "solar", "i"].dropna().astype(int).tolist())
GW = set(df_ativos.loc[df_ativos["ativo"] == "eolica", "i"].dropna().astype(int).tolist())
B  = set(df_ativos.loc[df_ativos["ativo"] == "bateria", "i"].dropna().astype(int).tolist())

# ============================================================
# 4) CONJUNTO DE LINHAS L E PARÂMETROS ELÉTRICOS
# ============================================================
df_linhas = df[df["tipo"] == "LINHA"].copy()
df_linhas["i"] = df_linhas["i"].astype(int)
df_linhas["j"] = df_linhas["j"].astype(int)

L = [(row.i, row.j) for row in df_linhas.itertuples(index=False)]

F_max = {(row.i, row.j): float(row.F_max) for row in df_linhas.itertuples(index=False)}
b     = {(row.i, row.j): float(row.b) for row in df_linhas.itertuples(index=False)}
R     = {(row.i, row.j): float(row.R) for row in df_linhas.itertuples(index=False)}
c_loss = {(row.i, row.j): float(row.c_loss) for row in df_linhas.itertuples(index=False)}

# ============================================================
# 5) DEMANDA D[i,t]
# ============================================================
df_dem = df[df["tipo"] == "DEMANDA"].copy()

D = {(i, t): 0.0 for i in N for t in T}
for row in df_dem.itertuples(index=False):
    i = int(row.i)
    t = int(row.t)
    if (i in N) and (t in T):
        D[(i, t)] = float(row.valor)

# ============================================================
# 6) DISPONIBILIDADE SOLAR E EÓLICA
# ============================================================
df_sol = df[df["tipo"] == "SOLAR"].copy()
df_eol = df[df["tipo"] == "EOLICA"].copy()

PS_avail = {(i, t): 0.0 for i in N for t in T}
PW_avail = {(i, t): 0.0 for i in N for t in T}

for row in df_sol.itertuples(index=False):
    i = int(row.i)
    t = int(row.t)
    if (i in N) and (t in T):
        PS_avail[(i, t)] = float(row.valor)

for row in df_eol.itertuples(index=False):
    i = int(row.i)
    t = int(row.t)
    if (i in N) and (t in T):
        PW_avail[(i, t)] = float(row.valor)

# ============================================================
# 7) CUSTO DE VERTIMENTO c_curt[i]
# ============================================================
df_curt = df[df["tipo"] == "CUSTO_CURT"].copy()

c_curt = {i: 0.0 for i in N}
for row in df_curt.itertuples(index=False):
    i = int(row.i)
    if i in N:
        c_curt[i] = float(row.valor)

# ============================================================
# 8) PARÂMETROS DAS BATERIAS
# ============================================================
df_bat = df[df["tipo"] == "BATERIA"].copy()

eta_ch   = {}
eta_dis  = {}
P_ch_max = {}
P_dis_max = {}
E_min    = {}
E_max    = {}
E0       = {}
c_ch     = {}
c_dis    = {}

for row in df_bat.itertuples(index=False):
    i = int(row.i)
    if i in B:
        eta_ch[i]    = float(row.eta_ch)
        eta_dis[i]   = float(row.eta_dis)
        P_ch_max[i]  = float(row.P_ch_max)
        P_dis_max[i] = float(row.P_dis_max)
        E_min[i]     = float(row.E_min)
        E_max[i]     = float(row.E_max)
        E0[i]        = float(row.E0)
        c_ch[i]      = float(row.c_ch)
        c_dis[i]     = float(row.c_dis)

# ============================================================
# 9) APROXIMAÇÃO POR PARTES DAS PERDAS
# ============================================================
# número de pontos de quebra
Kmax = 4
K = range(0, Kmax + 1)

f = {}
p = {}

for ell in L:
    for k in K:
        f[(k, ell)] = k * (F_max[ell] / Kmax)
        p[(k, ell)] = R[ell] * (f[(k, ell)] ** 2)

# ============================================================
# 10) ESCOLHA DA BARRA DE REFERÊNCIA
# ============================================================
i_ref = min(N)

# ============================================================
# 11) CHECAGEM RÁPIDA
# ============================================================
print("Barras N =", N)
print("Barras com solar GS =", GS)
print("Barras com eólica GW =", GW)
print("Barras com bateria B =", B)
print("Linhas L =", L)
print("Períodos =", list(T))
print("Barra de referência =", i_ref)

Barras N = [np.int64(1), np.int64(2), np.int64(3)]
Barras com solar GS = {1}
Barras com eólica GW = {2}
Barras com bateria B = {3}
Linhas L = [(1, 2), (2, 3)]
Períodos = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
Barra de referência = 1


2. Criando o problema e as variáveis em PuLP
Continuando no mesmo notebook:

In [2]:
# Criar problema de minimização
prob = pulp.LpProblem("Despacho_Economico_Microrrede", pulp.LpMinimize)

# 2.1 Variáveis renováveis (para todo t, i)
PS = pulp.LpVariable.dicts("PS", [(i, t) for i in N for t in T], lowBound=0)
PW = pulp.LpVariable.dicts("PW", [(i, t) for i in N for t in T], lowBound=0)
CS = pulp.LpVariable.dicts("CS", [(i, t) for i in N for t in T], lowBound=0)
CW = pulp.LpVariable.dicts("CW", [(i, t) for i in N for t in T], lowBound=0)

# 2.2 Variáveis da bateria (para i em B)
P_ch = pulp.LpVariable.dicts("P_ch", [(i, t) for i in B for t in T], lowBound=0)
P_dis = pulp.LpVariable.dicts("P_dis", [(i, t) for i in B for t in T], lowBound=0)
E = pulp.LpVariable.dicts("E", [(i, t) for i in B for t in T])

# 2.3 Ângulos de tensão
theta = pulp.LpVariable.dicts("theta", [(i, t) for i in N for t in T])

# 2.4 Fluxos de potência ativa nas linhas
F = pulp.LpVariable.dicts("F", [(l, t) for l in L for t in T])
F_pos = pulp.LpVariable.dicts("F_pos", [(l, t) for l in L for t in T], lowBound=0)
F_neg = pulp.LpVariable.dicts("F_neg", [(l, t) for l in L for t in T], lowBound=0)

# 2.5 Perdas e lambdas da aproximação por partes
Ploss = pulp.LpVariable.dicts("Ploss", [(l, t) for l in L for t in T], lowBound=0)
lam = pulp.LpVariable.dicts("lam", [(k, l, t) for k in K for l in L for t in T], lowBound=0)

3. Função objetivo (equação 1)
Equação (1) da proposta:

\begin{equation}
\begin{aligned}
\min \; Z = \sum_{t \in T} \bigg[
    &\sum_{i \in N} c^{\text{curt}}_i \big( C^{S}_{i,t} + C^{W}_{i,t} \big)+\;&\sum_{i \in B} \big( c^{\text{ch}}_i P^{\text{ch}}_{i,t} + c^{\text{dis}}_i P^{\text{dis}}_{i,t} \big)+\;&\sum_{\ell \in L} c^{\text{loss}}_\ell P^{\text{loss}}_{\ell,t}
\bigg] .
\end{aligned}
\end{equation}

In [3]:
# Função objetivo
prob += (
    pulp.lpSum(
        c_curt[i] * (CS[(i, t)] + CW[(i, t)]) 
        for i in N for t in T
    )
    +
    pulp.lpSum(
        c_ch[i] * P_ch[(i, t)] + c_dis[i] * P_dis[(i, t)]
        for i in B for t in T
    )
    +
    pulp.lpSum(
        c_loss[l] * Ploss[(l, t)]
        for l in L for t in T
    )
), "Custo_Total"

4. Restrições de geração renovável (5.5.1)
Equações (2)–(4) para solar, (5)–(7) para eólica:

Para todo $t \in T$, $i \in G^{S}$:
\begin{align}
0 \leq P^{S}_{i,t} \leq \overline{P}^{S,\text{avail}}_{i,t}, \\
0 \leq C^{S}_{i,t} \leq \overline{P}^{S,\text{avail}}_{i,t}, \\
P^{S}_{i,t} + C^{S}_{i,t} = \overline{P}^{S,\text{avail}}_{i,t}.
\end{align}

Para todo $t \in T$, $i \in G^{W}$:
\begin{align}
0 \leq P^{W}_{i,t} \leq \overline{P}^{W,\text{avail}}_{i,t}, \\
0 \leq C^{W}_{i,t} \leq \overline{P}^{W,\text{avail}}_{i,t}, \\
P^{W}_{i,t} + C^{W}_{i,t} = \overline{P}^{W,\text{avail}}_{i,t}.
\end{align}

In [4]:
# Solar
for t in T:
    for i in GS:
        prob += PS[(i, t)] <= PS_avail[(i, t)]
        prob += CS[(i, t)] <= PS_avail[(i, t)]
        prob += PS[(i, t)] + CS[(i, t)] == PS_avail[(i, t)]

# Eólica
for t in T:
    for i in GW:
        prob += PW[(i, t)] <= PW_avail[(i, t)]
        prob += CW[(i, t)] <= PW_avail[(i, t)]
        prob += PW[(i, t)] + CW[(i, t)] == PW_avail[(i, t)]

5. Dinâmica e limites da bateria (5.5.2)
Equações (8)–(11):

Para todo $t \in T$, $i \in B$:
\begin{equation}
E_{i,t+1} = E_{i,t} + \eta^{\text{ch}}_i P^{\text{ch}}_{i,t}\,\Delta t - \frac{1}{\eta^{\text{dis}}_i} P^{\text{dis}}_{i,t}\,\Delta t,
\end{equation}

\begin{equation}
\underline{E}_i \leq E_{i,t} \leq \overline{E}_i,
\end{equation}

\begin{equation}
0 \leq P^{\text{ch}}_{i,t} \leq \overline{P}^{\text{ch}}_i, \qquad
0 \leq P^{\text{dis}}_{i,t} \leq \overline{P}^{\text{dis}}_i.
\end{equation}

Condição inicial:
\begin{equation}
E_{i,1} = E_{i,0}.
\end{equation}

In [5]:
for i in B:
    # condição inicial
    prob += E[(i, min(T))] == E0[i]
    
    for t in T:
        # limites de potência
        prob += P_ch[(i, t)] <= P_ch_max[i]
        prob += P_dis[(i, t)] <= P_dis_max[i]
        
        # limites de energia
        prob += E_min[i] <= E[(i, t)]
        prob += E[(i, t)] <= E_max[i]
        
        # dinâmica (para t < último)
        if t < max(T):
            prob += (
                E[(i, t + 1)]
                == E[(i, t)]
                   + eta_ch[i] * P_ch[(i, t)] * Delta_t
                   - (1.0 / eta_dis[i]) * P_dis[(i, t)] * Delta_t
            )

6. Modelo DC de fluxo de potência (5.5.3)

Para cada linha $\ell = (i,j) \in L$ e período $t \in T$:
\begin{equation}
F_{\ell,t} = b_\ell \big( \theta_{i,t} - \theta_{j,t} \big).
\end{equation}

Limites de fluxo:
\begin{equation}
-\overline{F}_\ell \leq F_{\ell,t} \leq \overline{F}_\ell.
\end{equation}

Escolha da barra de referência:
\begin{equation}
\theta_{i^{\text{ref}},t} = 0 \quad \forall t \in T.
\end{equation}

In [6]:
i_ref = 1  # vamos tomar a barra 1 como referência

for t in T:
    # referência
    prob += theta[(i_ref, t)] == 0
    
    for (i, j) in L:
        prob += F[((i, j), t)] == b[(i, j)] * (theta[(i, t)] - theta[(j, t)])
        prob += F[((i, j), t)] <= F_max[(i, j)]
        prob += F[((i, j), t)] >= -F_max[(i, j)]

7. Balanço de potência ativa nas barras (5.5.4)

Defina os conjuntos de linhas incidentes em cada barra $i$:
- $\delta^{+}(i) = \{\ell = (i,j) \in L\}$: linhas saindo de $i$.
- $\delta^{-}(i) = \{\ell = (j,i) \in L\}$: linhas chegando a $i$.


In [7]:
delta_plus = {i: [] for i in N}
delta_minus = {i: [] for i in N}

for (u, v) in L:
    delta_plus[u].append((u, v))
    delta_minus[v].append((u, v))

Para todo $i \in N$, $t \in T$:
\begin{equation}
\begin{aligned}
&P^{S}_{i,t} + P^{W}_{i,t} + P^{\text{dis}}_{i,t} - P^{\text{ch}}_{i,t} - D_{i,t} &= \sum_{\ell \in \delta^{+}(i)} F_{\ell,t} - \sum_{\ell \in \delta^{-}(i)} F_{\ell,t} - \sum_{\ell \in \delta^{+}(i) \cup \delta^{-}(i)} P^{\text{loss}}_{\ell,t}.
\end{aligned}
\end{equation}

In [8]:
for t in T:
    for i in N:
        inj_ren = PS[(i, t)] + PW[(i, t)]
        inj_batt = 0
        if i in B:
            inj_batt = P_dis[(i, t)] - P_ch[(i, t)]
        
        # soma fluxos que saem e entram
        sum_out = pulp.lpSum(F[((l_i, l_j), t)] for (l_i, l_j) in delta_plus[i])
        sum_in  = pulp.lpSum(F[((l_i, l_j), t)] for (l_i, l_j) in delta_minus[i])
        
        # perdas nas linhas incidentes
        incident_lines = delta_plus[i] + delta_minus[i]
        sum_loss = pulp.lpSum(Ploss[((l_i, l_j), t)] for (l_i, l_j) in incident_lines)
        
        prob += (
            inj_ren + inj_batt - D[(i, t)]
            == sum_out - sum_in - sum_loss
        )

8. Aproximação por partes das perdas (5.5.5)

Decomposição do fluxo em partes positiva e negativa, para todo $\ell \in L$, $t \in T$:
\begin{align}
F_{\ell,t} &= F^{+}_{\ell,t} - F^{-}_{\ell,t}, \\
F^{+}_{\ell,t} &\geq 0, \qquad F^{-}_{\ell,t} \geq 0.
\end{align}

Módulo do fluxo aproximado por combinação convexa dos pontos de quebra:

\begin{equation}
F^{+}_{\ell,t} + F^{-}_{\ell,t} = \sum_{k \in K} f_{k,\ell}\, \lambda_{k,\ell,t},
\end{equation}

\begin{equation}
\sum_{k \in K} \lambda_{k,\ell,t} = 1, \qquad \lambda_{k,\ell,t} \geq 0.
\end{equation}

Perdas approximadas:
\begin{equation}
P^{\text{loss}}_{\ell,t} = \sum_{k \in K} p_{k,\ell}\, \lambda_{k,\ell,t}.
\end{equation}

In [9]:
for t in T:
    for l in L:
        # 16) decomposição
        prob += F[(l, t)] == F_pos[(l, t)] - F_neg[(l, t)]
        
        # 18) módulo aproximado por combinação convexa
        prob += (
            F_pos[(l, t)] + F_neg[(l, t)]
            == pulp.lpSum(f[(k, l)] * lam[(k, l, t)] for k in K)
        )
        
        # 19) convexidade das lambdas
        prob += pulp.lpSum(lam[(k, l, t)] for k in K) == 1
        
        # 20) perdas aproximadas
        prob += (
            Ploss[(l, t)]
            == pulp.lpSum(p[(k, l)] * lam[(k, l, t)] for k in K)
        )

9. Resolver e inspecionar resultados
Para fechar a primeira rodada:


In [10]:
status = prob.solve(pulp.PULP_CBC_CMD(msg=1))

print("Status:", pulp.LpStatus[status])
print("Valor ótimo:", pulp.value(prob.objective))

for t in T:
    print(f"\n--- Período {t} ---")
    for i in N:
        print(f"Barra {i}: PS={PS[(i,t)].varValue:.2f}, PW={PW[(i,t)].varValue:.2f}, "
              f"CS={CS[(i,t)].varValue:.2f}, CW={CW[(i,t)].varValue:.2f}")
    for l in L:
        print(f"Linha {l}: F={F[(l,t)].varValue:.2f}, Ploss={Ploss[(l,t)].varValue:.4f}")
    for i in B:
        print(f"Bateria barra {i}: E={E[(i,t)].varValue:.2f}, "
              f"P_ch={P_ch[(i,t)].varValue:.2f}, P_dis={P_dis[(i,t)].varValue:.2f}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/fernando/anaconda3/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/d7d368c9515d4caba479443f8f01f35e-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/d7d368c9515d4caba479443f8f01f35e-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 701 COLUMNS
At line 2883 RHS
At line 3580 BOUNDS
At line 3725 ENDATA
Problem MODEL has 696 rows, 864 columns and 1941 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 238 (-458) rows, 500 (-364) columns and 1262 (-679) elements
Perturbing problem by 0.001% of 91.700404 - largest nonzero change 0.00021532671 ( 0.067722467%) - largest zero change 0.00019152275
0  Obj -172.80129 Primal inf 721.93299 (97) Dual inf 13.082566 (48)
79  Obj 230.06449 Primal inf 121.91628 (51)
154  Obj 6400.4767 Primal inf 123.21632 (39)
219  Obj 6588.1697 P